# Father — post-mortem investigation

**Premise.** Monitoring reported an unexplained `sshd` restart and an unrecognised session on this
host. The system was powered off; disk and memory were acquired. Nothing about the cause is assumed
here — the mechanism has to come out of the evidence.

**Scope.** The preserved disk and memory acquisitions of one run, plus the prepared extractions
derived from them.

**Boundary.** Scenario execution records guided the design of this lab and are disclosed in
Section 7. They are not evidence: every finding must cite an acquired disk, timeline or RAM record.

In [ ]:
import json, os, re, sys
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

RUN_ID = "father-u22-20260913-01"

PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "shared" / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))
from investigations.common import forensics as fx

case = fx.load_case(PROJECT_ROOT, RUN_ID)
prepared = json.loads((case.prepared / "prepare.json").read_text())

# Work from the case directory so every command below reads as a short, quotable path.
os.chdir(case.run_root)
IMG = str(case.disk_image.relative_to(case.run_root))
OUT, DATA = Path("investigation/output"), Path("investigation/data")
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)


def product(name):
    "Path of a prepared product, echoing the invocation that produced it."
    p = prepared["products"][name]
    root = str(case.run_root) + "/"
    print("$ " + " ".join(a.replace(root, "") for a in p["argv"]))
    print(f"# recorded: {p['state']}, exit {p.get('exit_code')}")
    return Path(p["path"])


def istat_times(text):
    "The four inode times from istat output, as UTC timestamps."
    labels = {"atime": "Accessed", "mtime": "File Modified",
              "ctime": "Inode Modified", "crtime": "File Created"}
    out = {}
    for key, label in labels.items():
        m = re.search(rf"^{label}:\s*(.+?)\s*$", text, re.M)
        out[key] = pd.to_datetime(re.sub(r"\s+\([A-Z]+\)$", "", m.group(1)), utc=True) if m else pd.NaT
    return out


SECTOR_SIZE = int(re.search(r"Units are in (\d+)-byte sectors",
                            Path(prepared["products"]["mmls"]["path"]).read_text()).group(1))
ROOT = [(int(p["argv"][p["argv"].index("-o") + 1]), name)
        for name, p in prepared["products"].items()
        if name.startswith("fsstat-") and p["state"] == "ok"
        and "File System Type: Ext4" in Path(p["path"]).read_text()]
print(f"Ext4 candidates: {ROOT}")
ROOT_OFFSET, ROOT_FS_PRODUCT = ROOT[0]

## 0. Evidence and scope

### 0.1 What evidence is this, and when was it acquired?

In [ ]:
print(f"{case.manifest['scenario']} on {case.platform['guest_os']}, "
      f"kernel {case.platform['kernel']} ({case.platform['arch']}), "
      f"recorded timezone {case.timezone}")

fx.show(pd.DataFrame([
    {"source": src, "path": rec["path"], "bytes": rec["size_bytes"],
     "recorded_sha256": rec["sha256"], "started_at": rec["started_at"],
     "ended_at": rec["ended_at"]}
    for src, rec in (("disk", case.acquisition["disk"]), ("ram", case.acquisition["memory"]))
]), n=2, caption="Acquired evidence")

### 0.2 Is the evidence internally consistent?

The EWF verification ran once during preparation and is not repeated here.

In [ ]:
fx.sh(f"grep -E 'hash calculated|SUCCESS|FAILURE' {product('ewfverify')}",
      label="s0-02-ewfverify", out_dir=OUT)

### 0.3 What limits this examination?

RAM and disk were acquired at different moments, so they are two states, not one snapshot. The
guest runs a vanilla logging profile, so absent records are bounded negatives rather than evidence
of absence.

In [ ]:
RAM_ENDED = pd.to_datetime(case.acquisition["memory"]["ended_at"], utc=True)
DISK_STARTED = pd.to_datetime(case.acquisition["disk"]["started_at"], utc=True)
print(f"RAM capture ended  {RAM_ENDED.isoformat()}")
print(f"disk image started {DISK_STARTED.isoformat()}")
print(f"gap {(DISK_STARTED - RAM_ENDED).total_seconds():.3f} s — anything the guest wrote in that "
      f"window is on disk but not in memory")

### 0.4 How should timestamps be read?

All output below is displayed in UTC.

In [ ]:
tz_inode, _ = fx.resolve(IMG, ROOT_OFFSET, "/etc/timezone")
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {tz_inode}", label="s0-04-timezone", out_dir=OUT)

lt_inode, lt_chain = fx.resolve(IMG, ROOT_OFFSET, "/etc/localtime")
fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {lt_inode}",
      label="s0-04-localtime-istat", out_dir=OUT, tail=12)
print(f"/etc/localtime -> {lt_chain[-1]['path']}")
print(f"examiner timezone: {datetime.now().astimezone().tzinfo}")

**Interpretation.** _(to write)_

## 1. Disk: what happened on this filesystem, and what persists?

### 1.1 Which partition holds the root filesystem?

In [ ]:
fx.sh(f"cat {product('mmls')}", label="s1-01-mmls", out_dir=OUT)
fx.sh(f"head -n 30 {product(ROOT_FS_PRODUCT)}", label="s1-01-fsstat", out_dir=OUT)
print(f"root filesystem at sector {ROOT_OFFSET} ({SECTOR_SIZE}-byte sectors)")

### 1.2 What changed on this filesystem shortly before acquisition?

No mechanism is assumed. The bodyfile is queried for every allocated object whose mtime, ctime or
crtime falls in the 24 hours before the disk was imaged, ordered by time. The window is a stated
choice, bounded by the acquisition time; a longer window is one edit away.

In [ ]:
BODY = fx.load_bodyfile(product("allocated.body"))
ALLOCATED = BODY[BODY["name_state"].eq("allocated")]

WINDOW_START = DISK_STARTED - timedelta(hours=24)
in_window = ALLOCATED[
    ALLOCATED[["mtime_utc", "ctime_utc", "crtime_utc"]].ge(WINDOW_START).any(axis=1)
].copy()
in_window["last_change_utc"] = in_window[["mtime_utc", "ctime_utc", "crtime_utc"]].max(axis=1)
in_window = in_window.sort_values("last_change_utc")

in_window.to_csv(OUT / "s1-02-recent-changes.csv", index=False)
fx.show(in_window, cols=("last_change_utc", "name", "inode", "size", "mtime_utc",
                         "ctime_utc", "crtime_utc", "locator"),
        n=60, caption=f"Allocated objects changed after {WINDOW_START.isoformat()}")
print(f"{len(in_window)} objects in window; full result: {OUT}/s1-02-recent-changes.csv")

**Interpretation.** _(to write — which of these are ordinary boot/shutdown activity, and which are
not? note anything whose timestamps sit near or after the acquisition times)_

### 1.3 Which persistence-relevant paths exist, and do any of them appear above?

Scope of this sweep, stated so the negative is bounded: the dynamic-loader preload file and
configuration directory, cron, systemd system units, rc links, profile scripts, and per-user shell
startup files. Anything outside this list was not examined here.

In [ ]:
LOCATIONS = [
    ("dynamic loader preload",  r"^/etc/ld\.so\.preload$"),
    ("dynamic loader config",   r"^/etc/ld\.so\.conf\.d(?:/|$)"),
    ("cron",                    r"^/etc/(?:crontab$|cron(?:\.|/|$))"),
    ("systemd system units",    r"^/etc/systemd/system(?:/|$)"),
    ("rc links",                r"^/etc/rc[0-6S]\.d(?:/|$)"),
    ("profile scripts",         r"^/etc/profile\.d(?:/|$)"),
    ("shell startup",           r"^/(?:root|home/[^/]+)/\.(?:bashrc|profile)$"),
]
hits = pd.concat(
    [ALLOCATED[ALLOCATED["name"].str.match(p, na=False)].assign(location=name)
     for name, p in LOCATIONS],
    ignore_index=True,
)
hits["in_window"] = hits["locator"].isin(in_window["locator"])
hits.to_csv(OUT / "s1-03-persistence-paths.csv", index=False)

print(hits.groupby("location").agg(entries=("name", "size"),
                                   changed_in_window=("in_window", "sum")).to_string())
print(f"\nfull inventory: {OUT}/s1-03-persistence-paths.csv\n")

fx.show(hits[hits["in_window"]].sort_values("mtime_utc", ascending=False),
        cols=("location", "name", "inode", "mode", "size", "mtime_utc"),
        n=20, caption="Persistence-location entries that changed inside the window")

preload = hits[hits["location"].eq("dynamic loader preload")]
print(f"preload entries found: {len(preload)}")
PRELOAD_ROW = preload.iloc[0]
PRELOAD_PATH = str(PRELOAD_ROW["name"])

**Interpretation.** _(to write — a stock Ubuntu cloud image ships no `/etc/ld.so.preload`)_

### 1.4 What does that configuration file contain?

In [ ]:
preload_inode, _ = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_PATH)
preload_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {preload_inode}",
                      label="s1-04-preload-istat", out_dir=OUT, tail=20)
preload_icat = fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {preload_inode}",
                     label="s1-04-preload-icat", out_dir=OUT)

ENTRIES = [w for line in preload_icat.stdout.splitlines()
           for w in line.partition("#")[0].split()]
print(f"\nconfiguration entries: {ENTRIES}")
PRELOAD_OBJECT_PATH = ENTRIES[0]

**Interpretation.** _(to write)_

### 1.5 What is the referenced object?

In [ ]:
object_inode, object_chain = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_OBJECT_PATH)
print(" -> ".join(f"{e['path']}({e['inode']})" + (f" [link {e['target']}]" if "target" in e else "")
                  for e in object_chain))

object_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {object_inode}",
                     label="s1-05-object-istat", out_dir=OUT, tail=20)

OBJECT_BIN = DATA / "s1-05-referenced-object.bin"
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {object_inode}",
      label=OBJECT_BIN.name, out_dir=DATA, binary=True, show=False)

fx.sh(f"file {OBJECT_BIN}", label="s1-05-object-file", out_dir=OUT)
fx.sh(f"sha256sum {OBJECT_BIN}", label="s1-05-object-sha256", out_dir=OUT)
fx.sh(f"readelf -h -d {OBJECT_BIN}", label="s1-05-object-readelf", out_dir=OUT, tail=20)
strings = fx.sh(f"strings -a -t x {OBJECT_BIN}", label="s1-05-object-strings",
                out_dir=OUT, tail=0)
print(f"\n{len(strings.stdout.splitlines())} strings preserved in {OUT}/s1-05-object-strings.txt")

## 2. Surrounding activity: who, when, and what else changed?

The complete outputs from 1.2 and 1.5 are the pivot source. Account and time values come from
2.1; object strings are limited to recovered paths, authentication text and concealment markers.
Routine ELF symbols and scenario-supplied names are not search terms.

### 2.1 Which login sessions exist?

In [ ]:
SESSION_OUTPUTS = {"wtmp": "", "btmp": ""}
for name, reader in (("wtmp", "last"), ("btmp", "lastb")):
    guest_path = f"/var/log/{name}"
    rows = ALLOCATED[ALLOCATED["name"].eq(guest_path)]
    print(f"{guest_path}: {len(rows)} allocated bodyfile entries")
    if len(rows) != 1:
        continue
    inode = str(rows.iloc[0]["inode"])
    target = DATA / f"s2-01-{name}"
    extracted = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(guest_path)} {fx.q(IMG)} > {fx.q(target)}",
                      label=f"s2-01-{name}-fcat", out_dir=OUT)
    print(f"{guest_path} (inode {inode}): fcat exit {extracted.returncode}, {target.stat().st_size} bytes")
    if extracted.returncode != 0:
        continue
    viewed = fx.sh(f"TZ=UTC {reader} -F -i -f {fx.q(target)}",
                   label=f"s2-01-{reader}", out_dir=OUT, tail=40)
    SESSION_OUTPUTS[name] = viewed.stdout

excluded = {"reboot", "shutdown", "runlevel"}
session_lines = [line for text in SESSION_OUTPUTS.values() for line in text.splitlines()
                 if line.split() and " begins " not in line and line.split()[0] not in excluded]
SESSION_USERS = sorted({line.split()[0] for line in session_lines})
stamps = re.findall(r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun) \w{3} +\d+ \d{2}:\d{2}:\d{2} \d{4}",
                    "\n".join(session_lines))
SESSION_TIMES = sorted({pd.to_datetime(stamp, utc=True) for stamp in stamps})
TIME_PIVOTS = sorted({v for stamp in SESSION_TIMES
                      for v in (stamp.strftime("%b %e %H:%M"),
                                stamp.strftime("%Y-%m-%dT%H:%M"))})
print(f"session users: {SESSION_USERS}")
print(f"session time pivots: {TIME_PIVOTS}")

**Interpretation.** _(to write)_

### 2.2 What local accounts and privileges existed?

The shadow file is extracted for later byte comparison, but only its inode metadata is displayed.

In [ ]:
ACCOUNT_FILES = {}
for guest_path in ("/etc/passwd", "/etc/group", "/etc/shadow"):
    inode, _ = fx.resolve(IMG, ROOT_OFFSET, guest_path)
    target = DATA / f"s2-02-{Path(guest_path).name}"
    extracted = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(guest_path)} {fx.q(IMG)} > {fx.q(target)}",
                      label=f"s2-02-{target.name}-fcat", out_dir=OUT, tail=10)
    ACCOUNT_FILES[guest_path] = target
    print(f"{guest_path}: fcat exit {extracted.returncode}, {target.stat().st_size} bytes")
    if guest_path != "/etc/shadow":
        fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-02-{target.name}-numbered",
              out_dir=OUT, tail=80)

shadow_inode, _ = fx.resolve(IMG, ROOT_OFFSET, "/etc/shadow")
fx.sh(f"istat -o {fx.q(ROOT_OFFSET)} -z UTC {fx.q(IMG)} {fx.q(shadow_inode)}",
      label="s2-02-shadow-istat", out_dir=OUT, tail=20)
sudo_rows = ALLOCATED[ALLOCATED["name"].str.match(r"^/etc/sudoers(?:$|\.d(?:/|$))", na=False)]
fx.show(sudo_rows, cols=("name", "inode", "mode", "uid", "gid", "size", "locator"),
        n=40, caption="Sudo policy inventory")
for row in sudo_rows[sudo_rows["mode"].str.startswith("r/")].itertuples():
    target = DATA / f"s2-02-sudo-{row.inode}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-02-sudo-{row.inode}-fcat", out_dir=OUT, tail=10)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-02-sudo-{row.inode}-numbered",
          out_dir=OUT, tail=120)

**Interpretation.** _(to write)_

### 2.3 What do the authentication and service logs record?

The search terms are printed before use. They come from the sessions above, the initial `sshd`
trigger, and the already-enumerated object strings in 1.5.

In [ ]:
# Analyst selection, read from investigation/output/s1-05-object-strings.txt.
# Run-specific by nature: a different run needs this list re-read and rewritten.
SELECTED_STRINGS = [
    "AUTHENTICATE:", "lobster", "Enjoy the shell!", "__malicious_",
    "/proc/net/tcp", "/tmp/silly.txt", PRELOAD_PATH, PRELOAD_OBJECT_PATH,
]
SEARCH_PIVOTS = sorted(set(SESSION_USERS + ["sshd"]
                          + [s for s in SELECTED_STRINGS if s in strings.stdout]))
PIVOT_FILE = OUT / "s2-03-pivots.txt"
PIVOT_FILE.write_text("\n".join(SEARCH_PIVOTS) + "\n")
print("search pivots:\n" + "\n".join(f"  {v}" for v in SEARCH_PIVOTS))

log_rows = ALLOCATED[ALLOCATED["name"].str.match(
    r"^/var/log/(?:auth\.log$|syslog$|journal/.+\.journal$)", na=False)]
fx.show(log_rows, cols=("name", "inode", "size", "mtime_utc", "locator"),
        n=20, caption="Allocated logs selected for extraction")
LOG_FILES = {}
for row in log_rows.itertuples():
    target = DATA / f"s2-03-{row.inode}-{Path(row.name).name}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-03-{row.inode}-fcat", out_dir=OUT)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode == 0:
        LOG_FILES[row.name] = target

journal_files = [t for n, t in LOG_FILES.items() if n.endswith(".journal")]
print(f"journal files extracted: {len(journal_files)}")
LOG_SOURCES = {Path(name).name: target for name, target in LOG_FILES.items() if name in ("/var/log/auth.log", "/var/log/syslog")}
if journal_files:
    files = " ".join(f"--file={fx.q(path)}" for path in journal_files)
    rendered = fx.sh(f"journalctl --utc --no-pager -o short-iso-precise {files}",
                     label="s2-03-journal-rendered", out_dir=OUT, tail=0)
    print(f"journalctl exit {rendered.returncode}, {len(rendered.stdout.splitlines())} rendered lines")
    if rendered.returncode == 0:
        LOG_SOURCES["journal"] = OUT / "s2-03-journal-rendered.txt"
if (latest := max(SESSION_TIMES, default=None)) is not None:
    print(f"search day from 2.1: {latest.date().isoformat()} UTC")
    for name, source in LOG_SOURCES.items():
        day = latest.strftime("%Y-%m-%d" if name == "journal" else "%b %e")
        matched = fx.sh(f"grep -nF {fx.q(day)} {fx.q(source)} | grep -Fi -f {fx.q(PIVOT_FILE)}",
                        label=f"s2-03-{name}-matches", out_dir=OUT, tail=80)
        print(f"{name}: {len(matched.stdout.splitlines())} matching lines "
              f"(grep exit {matched.returncode})")
else:
    print("log search not run: 2.1 supplied zero session times")

**Interpretation.** _(to write)_

### 2.4 Is there shell history for any account?

Scope: allocated bodyfile entries named `.*history` directly under `/root` or an immediate
`/home/<user>` directory. A zero-row result is limited to those names and allocated entries; it
does not establish erasure.

In [ ]:
history_rows = ALLOCATED[ALLOCATED["name"].str.match(
    r"^/(?:root|home/[^/]+)/\.[^/]*history$", na=False
)]
print("scope searched: allocated.body; /root and immediate /home/<user>; names .*[Hh]istory")
fx.show(history_rows, cols=("name", "inode", "mode", "uid", "gid",
                             "size", "mtime_utc", "locator"),
        n=30, caption="Allocated shell-history candidates")
for row in history_rows.itertuples():
    target = DATA / f"s2-04-history-{row.inode}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-04-history-{row.inode}-fcat", out_dir=OUT, tail=10)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode == 0:
        fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-04-history-{row.inode}-numbered",
              out_dir=OUT, tail=80)

**Interpretation.** _(to write)_

### 2.5 Do the staging paths named in the logs exist, and what else is there?

The log lines recovered in 2.3 name specific paths written during the session. This block does
not assume where staging happened: it resolves each directory from the evidence and inventories
it in full — allocated and deleted separately — so that the log-named paths can be confirmed on
disk and anything the logs did **not** mention is also visible. All four listings are displayed
before anything is selected.

In [ ]:
STAGING_LISTINGS = {}
for guest_path in ("/tmp", "/dev/shm"):
    try:
        directory_inode, chain = fx.resolve(IMG, ROOT_OFFSET, guest_path)
    except fx.ResolveError as error:
        print(f"{guest_path}: not resolved; 0 inventories ({error})")
        continue
    print(f"{guest_path}: " + " -> ".join(
        f"{entry['path']}({entry['inode']})" +
        (f" [link {entry['target']}]" if "target" in entry else "")
        for entry in chain))
    for state, switch in (("allocated", "-u"), ("deleted", "-d")):
        label = f"s2-05-{guest_path.strip('/').replace('/', '-')}-{state}"
        listing = fx.sh(
            f"fls -r -p {switch} -o {fx.q(ROOT_OFFSET)} {fx.q(IMG)} {fx.q(directory_inode)}",
            label=label, out_dir=OUT, tail=80)
        STAGING_LISTINGS[(guest_path, state)] = listing.stdout
        print(f"{guest_path} {state}: {len(listing.stdout.splitlines())} entries, "
              f"fls exit {listing.returncode}")

**Interpretation.** _(to write)_

### 2.6 What are the staged files, and do any duplicate a system file?

Every regular file in the four inventories above is selected. `icat` preserves its bytes;
`file`, `sha256sum` and `strings` characterise it. Hash equality is checked against the extracted
account databases from 2.2; decoded text is never compared.

In [ ]:
fls_line = re.compile(
    r"^(?P<kind>\S+)\s+(?:\*\s*)?(?P<inode>\d+(?:-\d+-\d+)?)(?:\(realloc\))?:\s+(?P<name>.+)$")
staged = []
for (guest_path, state), text in STAGING_LISTINGS.items():
    for line in text.splitlines():
        match = fls_line.match(line)
        if match and match["kind"].startswith("r/"):
            staged.append({"directory": guest_path, "state": state, **match.groupdict()})
fx.show(pd.DataFrame(staged), n=40, caption="Regular files selected from the inventories")
original_hashes = {name: fx.sha256_file(path) for name, path in ACCOUNT_FILES.items()}
for index, item in enumerate(staged, 1):
    target = DATA / f"s2-06-{index:02d}-{item['state']}.bin"
    got = fx.sh(f"icat -o {fx.q(ROOT_OFFSET)} {fx.q(IMG)} {fx.q(item['inode'])} > {fx.q(target)}",
                label=f"s2-06-{index:02d}-icat", out_dir=OUT, tail=10)
    print(f"{item['directory']}/{item['name']} [{item['state']} inode {item['inode']}]: "
          f"icat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode != 0:
        continue
    fx.sh(f"file {fx.q(target)}", label=f"s2-06-{index:02d}-file", out_dir=OUT)
    fx.sh(f"sha256sum {fx.q(target)}", label=f"s2-06-{index:02d}-sha256", out_dir=OUT)
    fx.sh(f"strings -a -n 6 {fx.q(target)}", label=f"s2-06-{index:02d}-strings",
          out_dir=OUT, tail=40)
    digest = fx.sha256_file(target)
    matches = [n for n, d in original_hashes.items() if d == digest]
    print(f"byte-identical extracted system files: {matches or 'none'}")
    for name in matches:
        fx.sh(f"sha256sum {fx.q(target)} {fx.q(ACCOUNT_FILES[name])}",
              label=f"s2-06-{index:02d}-comparison", out_dir=OUT)

**Interpretation.** _(to write)_

## 3. Memory: what was running?

The prepared RAM products are loaded from their preserved JSON. Plugins absent from preparation
run inline against the acquired memory image and the recorded symbol directory.

### 3.1 Does the image match the symbols?

In [ ]:
MEM = os.path.relpath(case.memory_image, case.run_root)
ISF_DIR = os.path.relpath(case.isf_path.parent, case.run_root)
VOL_BASE = f"vol3 -f {fx.q(MEM)} -s {fx.q(ISF_DIR)} -r json"
BANNERS = fx.load_vol(product("banners.json"))
fx.show(BANNERS, cols=("Banner", "Offset", "locator"),
        n=len(BANNERS), caption="Prepared Linux banners")

**Interpretation.** _(to write)_

### 3.2 What processes existed, and how are they related?

In [ ]:
PSTREE = fx.load_vol(product("pstree.json"))
PSAUX = fx.load_vol(product("psaux.json"))
fx.show(PSTREE, cols=("PID", "PPID", "TID", "COMM",
                            "parent_locator", "locator"),
        n=50, caption="Prepared process tree")
fx.show(PSAUX, cols=("PID", "PPID", "COMM", "ARGS", "locator"),
        n=50, caption="Prepared process arguments")
print(f"pstree rows: {len(PSTREE)}; psaux rows: {len(PSAUX)}")

**Interpretation.** _(to write)_

### 3.3 Which processes map the object found in Section 1?

The filter is the final evidence-resolved path in `object_chain`, not a typed filename. Counts
use distinct PID and path pairs rather than individual virtual-memory mapping rows.

In [ ]:
MAPS = fx.load_vol(product("proc.Maps.json"))
OBJECT_MAP_PATH = object_chain[-1]["path"]
OBJECT_MAPS = MAPS[MAPS["File Path"].eq(OBJECT_MAP_PATH)].copy()
OBJECT_PAIRS = (OBJECT_MAPS[["PID", "Process", "File Path"]]
                .drop_duplicates(["PID", "File Path"])
                .sort_values(["PID", "File Path"]))
MAPPING_PIDS = OBJECT_PAIRS["PID"].dropna().astype(int).tolist()
print(f"Section 1 path: {PRELOAD_OBJECT_PATH}")
print(f"resolved map path: {OBJECT_MAP_PATH}; disk inode: {object_inode}")
fx.show(OBJECT_MAPS, cols=("PID", "Process", "Start", "End", "Flags",
                                  "Inode", "File Path", "locator"),
        n=40, caption="Mapping rows for the resolved object path")
fx.show(OBJECT_PAIRS, n=40, caption="Distinct PID and object-path pairs")
print(f"mapping rows: {len(OBJECT_MAPS)}; distinct PID + path pairs: "
      f"{len(OBJECT_PAIRS)}; PIDs: {MAPPING_PIDS}")

**Interpretation.** _(to write)_

### 3.4 Is `LD_PRELOAD` present in any process environment?

A zero-row result is expected for file-based preload configuration and is reported as a result.

In [ ]:
ENVARS_JSON = OUT / "s3-04-envars.json"
envars_run = fx.sh(f"{VOL_BASE} linux.envars.Envars > {fx.q(ENVARS_JSON)}",
                    label="s3-04-envars-run", out_dir=OUT, tail=20)
print(f"linux.envars exit {envars_run.returncode}")
ENVARS = pd.DataFrame()
ENV_PRELOAD = pd.DataFrame()
if envars_run.returncode == 0:
    ENVARS = fx.load_vol(ENVARS_JSON)
    if not ENVARS.empty:
        env_text = ENVARS.astype("string")
        mask = env_text.apply(
            lambda col: col.str.contains("LD_PRELOAD", regex=False, na=False)
        ).any(axis=1)
        ENV_PRELOAD = ENVARS[mask]
    fx.show(ENV_PRELOAD, n=40, caption="Environment rows containing LD_PRELOAD")
    print(f"environment rows: {len(ENVARS)}; LD_PRELOAD rows: {len(ENV_PRELOAD)}")

**Interpretation.** _(to write)_

### 3.5 Which endpoints were open at capture?

The prepared socket and open-file inventories are displayed in full before attribution. The
object string `/proc/net/tcp` came from 1.5: a connection hidden from the live host may still
appear in memory-resident kernel structures.

In [ ]:
SOCKSTAT = fx.load_vol(product("sockstat.json"))
LSOF = fx.load_vol(product("lsof.json"))
with pd.option_context("display.max_rows", None):
    fx.show(SOCKSTAT, n=len(SOCKSTAT), caption="Full prepared socket inventory")
    fx.show(LSOF, n=len(LSOF), caption="Full prepared open-file inventory")
MAPPED_SOCKETS = SOCKSTAT[SOCKSTAT["PID"].isin(MAPPING_PIDS)]
MAPPED_LSOF = LSOF[LSOF["PID"].isin(MAPPING_PIDS)]
fx.show(MAPPED_SOCKETS, n=len(MAPPED_SOCKETS),
        caption="Sockets attributed to object-mapping PIDs")
fx.show(MAPPED_LSOF, n=len(MAPPED_LSOF),
        caption="Open files attributed to object-mapping PIDs")
print(f"all sockets: {len(SOCKSTAT)}; attributed sockets: {len(MAPPED_SOCKETS)}; "
      f"all open files: {len(LSOF)}; attributed open files: {len(MAPPED_LSOF)}")

**Interpretation.** _(to write)_

### 3.6 Does memory hold shell history the disk does not?

Section 2.4 found zero allocated history files in its stated scope. Shell PIDs are selected from
the process inventory and restricted to processes that mapped the Section 1 object.

In [ ]:
shell_mask = PSAUX["COMM"].astype("string").str.match(r"(?:ba)?sh$", na=False)
SHELL_ROWS = PSAUX[shell_mask & PSAUX["PID"].isin(MAPPING_PIDS)]
fx.show(SHELL_ROWS, cols=("PID", "PPID", "COMM", "ARGS", "locator"),
        n=20, caption="Object-mapping shell processes")
SHELL_PIDS = SHELL_ROWS["PID"].dropna().astype(int).tolist()
print(f"shell PIDs selected from 3.2 and 3.3: {SHELL_PIDS}")
BASH = pd.DataFrame()
if SHELL_PIDS:
    pid_args = " ".join(fx.q(pid) for pid in SHELL_PIDS)
    bash_json = OUT / "s3-06-bash.json"
    bash_run = fx.sh(f"{VOL_BASE} linux.bash.Bash --pid {pid_args} > {fx.q(bash_json)}",
                     label="s3-06-bash-run", out_dir=OUT, tail=20)
    print(f"linux.bash exit {bash_run.returncode}")
    if bash_run.returncode == 0:
        BASH = fx.load_vol(bash_json)
        fx.show(BASH, n=100, caption="Recovered in-memory shell history")
        print(f"recovered shell-history rows: {len(BASH)}")
else:
    print("linux.bash not run: zero relevant shell PIDs")

**Interpretation.** _(to write)_

### 3.7 Can the object be recovered from memory?

The lowest PID in the enumerated mapping pairs is the reproducible selection rule. SHA-256 is
compared with the Section 1 disk extraction; equality and inequality are both meaningful.

In [ ]:
ELF_PID = min(MAPPING_PIDS, default=None)
ELF_DIR = DATA / "s3-07-elfs"
ELF_DIR.mkdir(parents=True, exist_ok=True)
ELFS = pd.DataFrame()
DUMPED_OBJECTS = []
print(f"mapping PID selected for linux.elfs: {ELF_PID}")
if ELF_PID is not None:
    elfs_json = OUT / "s3-07-elfs.json"
    elf_cmd = (f"vol3 -o {fx.q(ELF_DIR)} -f {fx.q(MEM)} -s {fx.q(ISF_DIR)} "
               f"-r json linux.elfs.Elfs --pid {fx.q(ELF_PID)} --dump")
    elf_run = fx.sh(f"{elf_cmd} > {fx.q(elfs_json)}",
                    label="s3-07-elfs-run", out_dir=OUT, tail=20)
    print(f"linux.elfs exit {elf_run.returncode}")
    if elf_run.returncode == 0:
        ELFS = fx.load_vol(elfs_json)
        object_elfs = ELFS[ELFS["File Path"].eq(OBJECT_MAP_PATH)]
        fx.show(object_elfs, n=40, caption="Recovered rows for the mapped object")
        outputs = object_elfs.get("File Output", pd.Series(dtype="string"))
        DUMPED_OBJECTS = [ELF_DIR / str(name) for name in outputs if pd.notna(name)]
        print(f"object dump files present: {len(DUMPED_OBJECTS)}")
        for index, dumped in enumerate(DUMPED_OBJECTS, 1):
            fx.sh(f"sha256sum {fx.q(OBJECT_BIN)} {fx.q(dumped)}",
                  label=f"s3-07-comparison-{index}", out_dir=OUT)
            print(f"byte-identical to disk object: "
                  f"{fx.sha256_file(OBJECT_BIN) == fx.sha256_file(dumped)}")

**Interpretation.** _(to write)_

### 3.8 Are kernel-level mechanisms clean?

Valid zero rows from the inline kernel checks are reported as results, not failures.

In [ ]:
LSMOD = fx.load_vol(product("lsmod.json"))
KMSG = fx.load_vol(product("kmsg.json"))
fx.show(LSMOD, n=len(LSMOD), caption="Full prepared module inventory")
print(f"prepared kmsg rows: {len(KMSG)}; displaying the final 80")
fx.show(KMSG.tail(80), n=80, caption="Final prepared kernel-log rows")
KERNEL_CHECKS = {}
plugins = (
    ("linux.malware.check_syscall.Check_syscall", "check-syscall"),
    ("linux.malware.modxview.Modxview", "modxview"),
    ("linux.ebpf.EBPF", "ebpf"),
)
for plugin_name, label in plugins:
    target = OUT / f"s3-08-{label}.json"
    run = fx.sh(f"{VOL_BASE} {plugin_name} > {fx.q(target)}",
                label=f"s3-08-{label}-run", out_dir=OUT, tail=20)
    print(f"{plugin_name} exit {run.returncode}")
    if run.returncode != 0:
        continue
    result = fx.load_vol(target)
    KERNEL_CHECKS[label] = result
    print(f"{label} rows: {len(result)}")
    fx.show(result, n=80, caption=f"{plugin_name} result")

**Interpretation.** _(to write)_

## 4. Deletion recovery

Deleted-entry enumeration, inode recovery, ext4 journal, then carving. Keep content, partial
content, metadata-only traces, bounded negatives and tool failures distinct.

**Not implemented.**

## 5. Chronology

`mactime` over the allocated bodyfile and the Plaso export, both scoped to the incident window,
merged into one sourced chronology. Acquisition and examiner artifacts get their own lane.

**Not implemented. Plaso required.**

## 6. Result tables

Findings are written here, by hand, from the observations above. Then the four tables and the
permitted counts, per [ai/RULES.md](../../ai/RULES.md).

**Not implemented.**

## 7. Validation against the controlled scenario

Compare the reconstruction with the run's command log: agreements, discrepancies, and what the
evidence could not show. Ground truth is an experimental reference, not a fourth source.

**Not implemented.**